In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import glob
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

#file path
input_files = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
if not os.path.isdir(input_files):
    input_files = '/kaggle/input/rogii-wellbore-geology-prediction'

#create horizontal and typewell wellid
all_horizontal_train = []
all_typewell_train = []
all_horizontal_test = []
all_typewell_test = []

hw_global_counter = 0
tw_global_counter = 0

base_path = Path(input_files)
hw_files = sorted(list(base_path.rglob('*__horizontal_well.csv')))

for hw_path in hw_files:
    if hw_path.name.startswith('.') or 'cache' in hw_path.parts:
        continue

    well_id = hw_path.name.split('__')[0]
    folder_type = hw_path.parent.name
    tw_path = hw_path.parent / f"{well_id}__typewell.csv"
    if not tw_path.exists():
        print(f"Warning: Typewell missing for well {well_id} in {folder_type}. Skipping pair.")
        continue

    try:
        # Load Horizontal Well File
        hw_df = pd.read_csv(hw_path)
        hw_df['well_id'] = well_id
        hw_df['well_type'] = 'horizontal'
        hw_df['row_order'] = np.arange(hw_global_counter, hw_global_counter + len(hw_df))
        hw_global_counter += len(hw_df)
        
        # Load Typewell File
        tw_df = pd.read_csv(tw_path)
        tw_df['well_id'] = well_id
        tw_df['well_type'] = 'typewell'
        tw_df['row_order'] = np.arange(tw_global_counter, tw_global_counter + len(tw_df))
        tw_global_counter += len(tw_df)

        if folder_type == 'train':
            all_horizontal_train.append(hw_df)
            all_typewell_train.append(tw_df)
        elif folder_type == 'test':
            all_horizontal_test.append(hw_df)
            all_typewell_test.append(tw_df)
            
    except Exception as e:
        print(f"Error reading file pair for well {well_id}: {e}")

final_hw_train = pd.concat(all_horizontal_train, ignore_index=True) if all_horizontal_train else pd.DataFrame()
final_tw_train = pd.concat(all_typewell_train, ignore_index=True) if all_typewell_train else pd.DataFrame()

final_hw_test = pd.concat(all_horizontal_test, ignore_index=True) if all_horizontal_test else pd.DataFrame()
final_tw_test = pd.concat(all_typewell_test, ignore_index=True) if all_typewell_test else pd.DataFrame()
#print final dataset
print(final_hw_train.shape)
print(final_tw_train.shape)
print(final_hw_test.shape)
print(final_tw_test.shape)

In [ ]:
#finding missing columns in train and test value
missing_tw_cols = set(final_tw_train.columns) - set(final_tw_test.columns)
missing_hw_cols = set(final_hw_train.columns) - set(final_hw_test.columns)

print("Typewell Target/Label Columns:", missing_tw_cols)
print("Horizontal Well Target/Label Columns:", missing_hw_cols)

In [ ]:
#print info
print(final_tw_train.info())
print(final_hw_train.info())
print(final_tw_test.info())
print(final_hw_test.info())

In [ ]:
import numpy as np
import pandas as pd

# merge two dataset
def strict_geological_merge(hw_master, tw_master, split='train'):
    merged_wells = []
    for well_id in hw_master['well_id'].unique():
        hw_well = hw_master[hw_master['well_id'] == well_id].copy()
        tw_well = tw_master[tw_master['well_id'] == well_id].copy()
        if tw_well.empty:
            merged_wells.append(hw_well)
            continue

        tw_well = tw_well.sort_values('TVT')
        if split == 'train':
            known_hw = hw_well.dropna(subset=['TVT'])
            if not known_hw.empty:
                base_tvt = tw_well['TVT'].min()
                offset = (known_hw['TVT'] - (base_tvt - hw_well['Z'])).mean()
                hw_well['computed_tvt_axis'] = base_tvt - hw_well['Z'] + offset
            else:
                hw_well['computed_tvt_axis'] = hw_well['Z']
        else: 
            z_center_hw = hw_well['Z'].median()
            tvt_center_tw = tw_well['TVT'].median()
            hw_well['computed_tvt_axis'] = tvt_center_tw - (hw_well['Z'] - z_center_hw)

        closest_indices = np.abs(hw_well['computed_tvt_axis'].values[:, None] - tw_well['TVT'].values).argmin(axis=1)
        matched_tw_features = tw_well.iloc[closest_indices].copy().reset_index(drop=True)
        matched_tw_features = matched_tw_features.rename(columns={
            'GR': 'tw_reference_GR',
            'row_order': 'tw_source_row_order'
        }).drop(columns=['well_id', 'well_type'], errors='ignore')

        hw_well = hw_well.reset_index(drop=True)
        combined_well = pd.concat([hw_well, matched_tw_features], axis=1)
        merged_wells.append(combined_well)

    return pd.concat(merged_wells, ignore_index=True)

final_train_clean = strict_geological_merge(final_hw_train, final_tw_train, split='train')
final_test_clean  = strict_geological_merge(final_hw_test, final_tw_test, split='test')


In [ ]:
print("--- Row Count Integrity Check ---")
print(f"Original HW Train Rows: {len(final_hw_train)} | Merged Train Rows: {len(final_train_clean)}")
print(f"Original HW Test Rows : {len(final_hw_test)}  | Merged Test Rows : {len(final_test_clean)}")

assert len(final_hw_train) == len(final_train_clean), "Row count mismatch in Training Set!"
assert len(final_hw_test) == len(final_test_clean), "Row count mismatch in Testing Set!"
print("\nSuccess! Datasets merged side-by-side with 0 row inflation.")

In [ ]:
final_train_clean.info()


In [ ]:
final_train_clean.isnull().sum()


In [ ]:
#checking percentage of missing value
percentage = ((final_train_clean.isnull().sum()*100)/len(final_train_clean)).round(2)
print(percentage)

In [ ]:
import numpy as np
import pandas as pd

def fill_tvt_geometrically(df):
    df = df.copy()
    known_mask = df['TVT_input'].notna()
    if not known_mask.any():
        print("No anchor found! Falling back to raw Z tracking.")
        df['TVT_filled'] = df['Z']
        return df

    anchor_row = df[known_mask].iloc[0]
    anchor_tvt = anchor_row['TVT_input']
    anchor_z   = anchor_row['Z']
    df['TVT_filled'] = anchor_tvt - (df['Z'] - anchor_z)
    df['TVT_input'] = df['TVT_input'].fillna(df['TVT_filled'])
    df = df.drop(columns=['TVT_filled'])
    return df

final_train_clean = fill_tvt_geometrically(final_train_clean)
final_test_clean  = fill_tvt_geometrically(final_test_clean)

In [ ]:
print((final_train_clean.isnull().sum()*100)/len(final_train_clean))

In [ ]:
initial_missing = final_train_clean['GR'].isna().sum()

final_train_clean = final_train_clean.sort_values(['well_id', 'MD']).reset_index(drop=True)
final_train_clean['GR'] = final_train_clean.groupby('well_id')['GR'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)
still_missing = final_train_clean['GR'].isna().sum()
if still_missing > 0:
    final_train_clean['GR'] = final_train_clean['GR'].fillna(final_train_clean['tw_reference_GR'])
    global_mean_gr = final_train_clean['GR'].mean()
    final_train_clean['GR'] = final_train_clean['GR'].fillna(global_mean_gr)

In [ ]:
print((final_train_clean.isnull().sum()*100)/len(final_train_clean))

In [ ]:
initial_missing_geology = final_train_clean['Geology'].isna().sum()
final_train_clean = final_train_clean.sort_values(['well_id', 'MD']).reset_index(drop=True)
final_train_clean['Geology'] = final_train_clean.groupby('well_id')['Geology'].transform(
    lambda x: x.ffill().bfill()
)

still_missing_geology = final_train_clean['Geology'].isna().sum()
if still_missing_geology > 0:
    most_frequent_layer = final_train_clean['Geology'].mode()[0]
    final_train_clean['Geology'] = final_train_clean['Geology'].fillna(most_frequent_layer)

In [ ]:
final_train_clean['GR_roll_mean_5'] = (
    final_train_clean.groupby('well_id')['GR']
    .transform(lambda x: x.rolling(5, center=True).mean())
    .bfill()
    .ffill()
)
final_train_clean['GR_gradient'] = final_train_clean.groupby('well_id')['GR'].diff().fillna(0)
final_train_clean['inclination_rad'] = np.arcsin(final_train_clean['Z']/(final_train_clean['MD']))
final_train_clean['inclination_deg'] = np.degrees(final_train_clean['inclination_rad'])

In [ ]:
print((final_train_clean.isnull().sum()*100)/len(final_train_clean))

In [ ]:
print(final_train_clean.shape)

In [ ]:
formation_cols_to_drop = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

final_train_clean = final_train_clean.drop(columns=formation_cols_to_drop, errors='ignore')

print(f"Final Train Row Count: {len(final_train_clean)} (Should be exactly 5092255)")
print(f"Remaining Missing Values in Train Matrix:\n{final_train_clean.isna().sum()}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


train_df = final_train_clean.dropna(subset = ['TVT']).copy()
features = [
    'MD',               
    'Z',                
    'GR',               
    'tw_reference_GR',  
    'computed_tvt_axis',
    'GR_roll_mean_5',
    'GR_gradient',
    'inclination_deg'
]
X_final = train_df[features].values.astype(float)
scaler_x = MinMaxScaler()
X_scaled = scaler_x.fit_transform(X_final)

exact_row_count = int(X_scaled.shape[0]) 

Y_raw = train_df['TVT'].values

if len(Y_raw.shape) > 1:
    # Slices matching rows from the first column if multi-column
    Y_final = Y_raw[:exact_row_count, 0].reshape(-1, 1).astype(float)
else:
 
    Y_final = Y_raw[:exact_row_count].reshape(-1, 1).astype(float)

scaler_y = MinMaxScaler()
Y_scaled = scaler_y.fit_transform(Y_final)

print(f"X_scaled row length: {X_scaled.shape[0]}")
print(f"Y_scaled row length: {Y_scaled.shape[0]}")

x_train, x_val, y_train, y_val = train_test_split(
    X_scaled, Y_scaled, test_size=0.20, random_state=42
)


In [ ]:
from lightgbm import LGBMRegressor

print("Training baseline LightGBM Regressor...")



model = LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=1000,          # Boost max trees, but rely on early stopping
    learning_rate=0.03,          # Lowered learning rate for smoother convergence
    num_leaves=127,             
    max_depth=9,                
    min_child_samples=50,       
    reg_alpha=0.1,              # L1 Regularization to prune weak features
    reg_lambda=1.0,             # L2 Regularization to stabilize weights
    random_state=42,
    n_jobs=-1
)

model.fit(X_scaled,Y_scaled.ravel())
print("Model training complete!")
y_val_pred = model.predict(x_val)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print('RMSE value for Lightgbm:',rmse)
print('R2 score for Lightgbm:',r2)

**Lightgbm** RMSE value for Lightgbm: 32.39429181428618
R2 score for Lightgbm: 0.997437040452172


In [ ]:
final_test_clean['GR_roll_mean_5'] = (
    final_test_clean.groupby('well_id')['GR']
    .transform(lambda x: x.rolling(5, center=True).mean())
    .bfill()
    .ffill()
)
final_test_clean['GR_gradient'] = final_test_clean.groupby('well_id')['GR'].diff().fillna(0)
final_test_clean['inclination_rad'] = np.arcsin(final_test_clean['Z']/(final_test_clean['MD']))
final_test_clean['inclination_deg'] = np.degrees(final_test_clean['inclination_rad'])



FEATURES = ['MD', 'Z', 'GR', 'tw_reference_GR', 'computed_tvt_axis','GR_roll_mean_5',
    'GR_gradient',
    'inclination_deg']
X_test_raw = final_test_clean[FEATURES].values.astype(float)
X_test_scaled_df = pd.DataFrame(
    scaler_x.transform(X_test_raw), 
    columns=FEATURES
)
scaled_predictions = model.predict(X_test_scaled)
predictions_2d = scaled_predictions.reshape(-1, 1)
real_world_predictions = scaler_y.inverse_transform(predictions_2d).ravel()

final_test_clean['tvt_pred'] = real_world_predictions


sample_sub = pd.read_csv('/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv')

sample_sub['well'] = sample_sub['id'].str.split('_').str[0]
sample_sub['row_idx'] = sample_sub['id'].str.split('_').str[1].astype(int)

final_test_clean = final_test_clean.sort_values(['well_id', 'MD']).reset_index(drop=True)
final_test_clean['row_idx'] = final_test_clean.groupby('well_id').cumcount()

prediction_lookup = {}
for idx, row in final_test_clean.iterrows():
    key = (str(row['well_id']).strip(), int(row['row_idx']))
    prediction_lookup[key] = float(row['tvt_pred'])
submission_rows = []
for _, row in sample_sub.iterrows():
    lookup_key = (str(row['well']), int(row['row_idx']))
    tvt_val = prediction_lookup.get(lookup_key, final_test_clean['tvt_pred'].median())
    
    submission_rows.append({'id': row['id'], 'tvt': tvt_val})

submission_df = pd.DataFrame(submission_rows)

submission_df.to_csv('submission.csv', index=False)
print(f"Submission successfully generated! Total Rows: {len(submission_df)}")

In [ ]:
for idx, row in submission_df.head(4).iterrows():
    print(f"{row['id']},{row['tvt']:.1f}")